# GraphCut-Q: Hardware-Aware QAOA for Weighted Max-Cut

**Qiskit Fall Fest 2026 · Quantum Optimization**

This notebook is a reproducible walkthrough of the existing GraphCut-Q project. It imports the project's `qaoa_solver.py` rather than creating a second QAOA implementation, checks the classical baseline, loads the saved benchmark CSVs, and displays the generated figures.

> **Important:** Run this notebook from the `D:\HACKATHON\GraphCut-Q` project directory so the Python modules, CSVs, and `figures/` folder can be found. The reported optimum of 27 applies to the six-vertex development graph currently in `qaoa_solver.py`; verify that this is the organizer-approved graph before submission.

## 1. Imports and project files

The notebook uses the project's existing environment. If a dependency is missing, install the project requirements in the activated virtual environment.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

PROJECT = Path.cwd()
print("Notebook working directory:", PROJECT)
print("Python:", sys.version.split()[0])

required_files = [
    "qaoa_solver.py",
    "classical_solver.py",
    "qaoa_strong_summary.csv",
    "qaoa_strong_seed_results.csv",
    "processor_benchmark_results.csv",
]
missing = [name for name in required_files if not (PROJECT / name).exists()]
if missing:
    print("Missing expected files:", missing)
    print("Change the notebook working directory to your GraphCut-Q project folder.")
else:
    print("Core project files found.")

## 2. Problem: weighted Max-Cut

Given a graph whose edges have weights, assign every vertex to one of two groups. An edge contributes its weight when its endpoints are assigned to different groups. The goal is to maximize the sum of weights of crossing edges.

For a bitstring `x`, the objective is

\[
C(x) = \sum_{(i,j)\in E} w_{ij}\,[x_i \ne x_j].
\]

The project has six vertices and ten weighted edges.

In [ ]:
from qaoa_solver import (
    NUM_VERTICES,
    EDGES,
    P,
    cut_weight,
    find_exact_optimum,
    build_qaoa_circuit,
)

print(f"Vertices: {NUM_VERTICES}")
print(f"Edges: {len(EDGES)}")
print(f"Configured QAOA depth P: {P}")
display(pd.DataFrame(EDGES, columns=["u", "v", "weight"]))

## 3. QUBO and cost Hamiltonian

For each edge `(i,j)` with weight `w`, the cut contribution is `w(x_i + x_j - 2 x_i x_j)`. Therefore, maximizing the cut is equivalent to minimizing the negative of this expression:

$$Q(x) = -\sum_{(i,j)\in E} w_{ij}(x_i+x_j-2x_ix_j).$$

Using $x_i=(1-Z_i)/2$, the cost to maximize can be represented by the Hamiltonian

$$H_C=\sum_{(i,j)\in E}\frac{w_{ij}}{2}(I-Z_iZ_j).$$

The implementation below builds the QUBO matrix and lists the weighted $Z_iZ_j$ terms for the exact graph configured in `qaoa_solver.py`.

In [ ]:
import numpy as np

Q = np.zeros((NUM_VERTICES, NUM_VERTICES), dtype=float)
for i, j, w in EDGES:
    Q[i, i] -= w
    Q[j, j] -= w
    Q[i, j] += 2 * w
    Q[j, i] += 2 * w

print('QUBO matrix for minimizing -cut weight:')
display(pd.DataFrame(Q).astype(int))
print('Cost Hamiltonian terms (identity offset omitted):')
for i, j, w in EDGES:
    print(f'  -{w}/2 * Z_{i} Z_{j}  (constant offset +{w}/2)')

def qubo_energy(bits):
    x = np.asarray(bits, dtype=float)
    return float(x @ Q @ x / 2)

for bits in optimal_solutions if 'optimal_solutions' in globals() else []:
    pass
print('Check: QUBO energy equals negative cut weight for sample 010110:',
      qubo_energy([0, 1, 0, 1, 1, 0]),
      ' | negative cut weight:', -cut_weight([0, 1, 0, 1, 1, 0]))

## 4. Classical baseline

The classical solver enumerates all \(2^6 = 64\) possible assignments. This provides an exact reference optimum for this small development instance.

In [ ]:
optimal_solutions, exact_optimum = find_exact_optimum()
print("Assignments checked:", 2 ** NUM_VERTICES)
print("Exact optimum cut weight:", exact_optimum)
print("Number of optimal assignments:", len(optimal_solutions))

for bits in optimal_solutions:
    group_a = [i for i, bit in enumerate(bits) if bit == 0]
    group_b = [i for i, bit in enumerate(bits) if bit == 1]
    print("Bitstring:", "".join(map(str, bits)))
    print("Group A:", group_a)
    print("Group B:", group_b)
    print()

## 5. Inspect the QAOA circuit

The circuit builder in `qaoa_solver.py` prepares a uniform superposition, applies a weighted cost layer using `RZZ` gates, and then applies an `RX` mixer layer. The parameters are optimized by the existing solver; this notebook does not rerun the optimization automatically.

In [ ]:
# A simple illustrative circuit using one gamma and one beta.
# This is for circuit inspection only, not a claim that these are optimized parameters.
example_circuit = build_qaoa_circuit(gammas=[0.5] * P, betas=[0.25] * P)
display(example_circuit.draw("mpl"))

## 6. Hardware-aware Processor B benchmark

The saved processor benchmark records the compiled circuit's resource counts. This is a result from the project's current compilation path; do not assume it is directly comparable to every other CSV unless the circuit and transpilation settings are identical.

In [ ]:
processor_path = PROJECT / "processor_benchmark_results.csv"
if processor_path.exists():
    processor = pd.read_csv(processor_path)
    display(processor)
else:
    print("processor_benchmark_results.csv not found.")

## 7. Strong benchmark: frozen parameters and holdout seeds

The strong benchmark summary is the main comparison to use for the headline depth-versus-quality results. It reports the same frozen parameters evaluated across 30 holdout seeds with 4,096 shots per seed, according to the project run notes.

Do not merge this table with the depth, trade-off, or validation CSVs: those are separate experiments and use different parameter choices.

In [ ]:
summary_path = PROJECT / "qaoa_strong_summary.csv"
if summary_path.exists():
    strong = pd.read_csv(summary_path).sort_values("p")
    cols = [
        "p", "fixed_expected_cut", "approximation_ratio",
        "ideal_mean", "ideal_sd", "noisy_mean", "noisy_sd",
        "noise_gap", "cx_count", "circuit_depth",
        "noisy_optimal_probability"
    ]
    display(strong[[c for c in cols if c in strong.columns]].round(4))
else:
    strong = None
    print("qaoa_strong_summary.csv not found.")

## 8. Results: cut quality

This plot compares ideal and noisy mean cut weight by QAOA depth. The dashed line marks the exact classical optimum of 27 for the current graph.

In [ ]:
if strong is not None:
    x = strong["p"].astype(str)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(x, strong["ideal_mean"], yerr=strong["ideal_sd"],
                marker="o", capsize=4, linewidth=2, label="Ideal simulation")
    ax.errorbar(x, strong["noisy_mean"], yerr=strong["noisy_sd"],
                marker="s", capsize=4, linewidth=2, label="Noisy simulation")
    ax.axhline(exact_optimum, linestyle="--", linewidth=1.5,
               label=f"Exact optimum = {exact_optimum}")
    ax.set(title="QAOA cut quality vs depth", xlabel="QAOA depth p",
           ylabel="Mean cut weight")
    ax.grid(True, alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 9. Results: noise penalty and resource cost

In [ ]:
if strong is not None:
    x = strong["p"].astype(str)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x, strong["noise_gap"], marker="o", linewidth=2)
    ax.set(title="Noise penalty vs QAOA depth", xlabel="QAOA depth p",
           ylabel="Ideal mean − noisy mean")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax1.plot(x, strong["cx_count"], marker="o", linewidth=2, label="CX gates")
    ax1.set_xlabel("QAOA depth p")
    ax1.set_ylabel("CX gate count")
    ax1.grid(True, alpha=0.25)
    ax2 = ax1.twinx()
    ax2.plot(x, strong["circuit_depth"], marker="s", linewidth=2, label="Circuit depth")
    ax2.set_ylabel("Circuit depth")
    ax1.set_title("Circuit resources vs QAOA depth")
    lines = ax1.get_lines() + ax2.get_lines()
    ax1.legend(lines, [line.get_label() for line in lines], loc="upper left")
    plt.tight_layout()
    plt.show()

## 10. Load the generated figures

These are the PNG files created by `plot_results.py`. This cell displays them inside the notebook when the `figures/` directory is present.

In [ ]:
figure_names = [
    "strong_mean_cut.png",
    "noise_penalty.png",
    "resource_cost.png",
    "optimal_probability.png",
    "noise_sensitivity.png",
]

for name in figure_names:
    path = PROJECT / "figures" / name
    print(name, "—", "found" if path.exists() else "not found")
    if path.exists():
        display(Image(filename=str(path)))

## 11. Interpretation

Use the **strong benchmark** as the primary depth comparison:

- In this experiment, p = 3 produced the highest observed ideal and noisy mean cut weight.
- Circuit cost rose from 35 CX gates and depth 52 at p = 1 to 120 CX gates and depth 151 at p = 3.
- The ideal-to-noisy mean gap also grew with depth, showing the trade-off between approximation quality and noise sensitivity.
- p = 1 is cheaper; p = 3 performed better in this particular noisy simulation. The appropriate choice depends on the hardware constraints and target objective.

## 12. Limitations and submission checks

- These are simulator results, not quantum hardware results.
- Do not claim quantum advantage from this benchmark.
- Verify the six-vertex graph against the official challenge instance.
- Check the official Processor A topology carefully: the supplied challenge materials reportedly conflict, and a five-qubit processor cannot directly host six logical qubits without a different strategy.
- Confirm Processor D requirements and ensure all processor comparisons use the same graph and clearly documented compilation settings.
- Record Python/Qiskit versions, random seeds, shot count, transpilation optimization level, and the exact noise model in the final submission.